In [1]:
pwd

'/files/private/notebooks/printer_ml/notebooks'

In [2]:
cd /files/private/notebooks/printer_ml

/files/private/notebooks/printer_ml


In [3]:
import sys
print(sys.executable)

/files/private/conda/mamba_env/bin/python


In [15]:
!/usr/bin/nvidia-smi


Failed to initialize NVML: Unknown Error


In [24]:
!git pull

Already up to date.


In [22]:
# Step 1: extract + cache frozen DINOv2 embeddings (clean + augmented variants for rare-target videos)
# Run this once. Re-run only if you change CFG inside extract_dinov2_embeddings_kfold.py
import os

os.environ["OMP_NUM_THREADS"] = "1"
os.environ["OPENBLAS_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"
os.environ["VECLIB_MAXIMUM_THREADS"] = "1"
os.environ["NUMEXPR_NUM_THREADS"] = "1"

import sys

!{sys.executable} scripts/extract_dinov2_embeddings_kfold.py

DINOv2 K-fold embedding extraction (with augmented variants)
Device: cuda
Split dir: /files/private/notebooks/printer_ml/data/processed/kfold_splits
Output dir: /files/private/notebooks/printer_ml/results/dinov2_embeddings
Using cache found in /home/jovyan/.cache/torch/hub/facebookresearch_dinov2_main
/home/jovyan/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/swiglu_ffn.py:51: UserWarning: xFormers is not available (SwiGLU)
  warnings.warn("xFormers is not available (SwiGLU)")
/home/jovyan/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/attention.py:33: UserWarning: xFormers is not available (Attention)
  warnings.warn("xFormers is not available (Attention)")
/home/jovyan/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/block.py:40: UserWarning: xFormers is not available (Block)
  warnings.warn("xFormers is not available (Block)")
Global bin edges (log scale): [0.28210533 0.57668717 0.87126901 1.16585086 1.4604327 ]

Fold 1/5
Extracting train embedd

In [32]:
# Step 2: train the Mamba sequence head on the cached embeddings
# (oversampling via WeightedRandomSampler + embedding-space augmentation happen inside this script)
import os

os.environ["OMP_NUM_THREADS"] = "1"
os.environ["OPENBLAS_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"
os.environ["VECLIB_MAXIMUM_THREADS"] = "1"
os.environ["NUMEXPR_NUM_THREADS"] = "1"

import sys

!{sys.executable} scripts/train_mamba_cached_kfold.py

Mamba cached k-fold training (augmented + oversampled)
Device: cuda
Embeddings dir: /files/private/notebooks/printer_ml/results/dinov2_embeddings
Out dir: /files/private/notebooks/printer_ml/results/mamba_cached_kfold

Fold 1 | train rows (incl. variants): 1 | val videos: 18
Fold 1 | Epoch 001 | train_loss=0.21670 | val_mae=0.67668 | best=0.67668 | no_improve=0/10
Fold 1 | Epoch 002 | train_loss=0.05904 | val_mae=0.55803 | best=0.55803 | no_improve=0/10
Fold 1 | Epoch 003 | train_loss=0.02039 | val_mae=0.41654 | best=0.41654 | no_improve=0/10
Fold 1 | Epoch 004 | train_loss=0.03624 | val_mae=0.42900 | best=0.41654 | no_improve=1/10
Fold 1 | Epoch 005 | train_loss=0.00809 | val_mae=0.51453 | best=0.41654 | no_improve=2/10
Fold 1 | Epoch 006 | train_loss=0.00887 | val_mae=0.47230 | best=0.41654 | no_improve=3/10
Fold 1 | Epoch 007 | train_loss=0.00274 | val_mae=0.43732 | best=0.41654 | no_improve=4/10
Fold 1 | Epoch 008 | train_loss=0.00312 | val_mae=0.43544 | best=0.41654 | no_improve=5

In [9]:
!git config --global user.email "you@example.com"
!git config --global user.name "Your Name"

In [10]:
!git add .
!git commit -m "mamba_cached_results"
!git push

[main 2557a0f] mamba_cached_results
 112 files changed, 7173 insertions(+), 12 deletions(-)
 create mode 100644 results/dinov2_embeddings/config.json
 create mode 100644 results/dinov2_embeddings/embedding_summary_all_folds.json
 create mode 100644 results/dinov2_embeddings/fold_1/embedding_summary.json
 create mode 100644 results/dinov2_embeddings/fold_1/train_embeddings.pt
 create mode 100644 results/dinov2_embeddings/fold_1/val_embeddings.pt
 create mode 100644 results/dinov2_embeddings/fold_1/variant_plan.json
 create mode 100644 results/dinov2_embeddings/fold_2/embedding_summary.json
 create mode 100644 results/dinov2_embeddings/fold_2/train_embeddings.pt
 create mode 100644 results/dinov2_embeddings/fold_2/val_embeddings.pt
 create mode 100644 results/dinov2_embeddings/fold_2/variant_plan.json
 create mode 100644 results/dinov2_embeddings/fold_3/embedding_summary.json
 create mode 100644 results/dinov2_embeddings/fold_3/train_embeddings.pt
 create mode 100644 results/dinov2_embed

In [15]:
import pandas as pd
import numpy as np
for fold in range(1,6):
    df = pd.read_csv(f'data/processed/kfold_splits/fold_{fold}_train.csv')
    bins = pd.cut(df['axial_resolution'], bins=4, duplicates='drop')
    print(bins.value_counts().sort_index())
    print()

axial_resolution
(1.323, 2.071]     1
(2.071, 2.817]     9
(2.817, 3.562]    19
(3.562, 4.308]    39
Name: count, dtype: int64

axial_resolution
(2.07, 2.631]      4
(2.631, 3.19]     13
(3.19, 3.749]     23
(3.749, 4.308]    29
Name: count, dtype: int64

axial_resolution
(1.323, 2.071]     1
(2.071, 2.817]     8
(2.817, 3.562]    20
(3.562, 4.308]    40
Name: count, dtype: int64

axial_resolution
(1.323, 2.062]     1
(2.062, 2.799]     8
(2.799, 3.535]    20
(3.535, 4.272]    40
Name: count, dtype: int64

axial_resolution
(1.323, 2.071]     1
(2.071, 2.817]     7
(2.817, 3.562]    21
(3.562, 4.308]    40
Name: count, dtype: int64



In [26]:
!git add .
!git commit -m "mamba cached reggression results"
!git push

[main b181fbd] mamba cached reggression results
 111 files changed, 4932 insertions(+), 5923 deletions(-)
Enumerating objects: 170, done.
Counting objects: 100% (170/170), done.
Delta compression using up to 128 threads
Compressing objects: 100% (132/132), done.
Writing objects: 100% (132/132), 27.98 MiB | 12.75 MiB/s, done.
Total 132 (delta 21), reused 0 (delta 0), pack-reused 0
remote: Resolving deltas: 100% (21/21), completed with 8 local objects.
To https://github.com/4rcturi4n/printer_ml.git
   94d2923..b181fbd  main -> main
